In [1]:
library(googledrive)
drive_auth()

Is it OK to cache OAuth access credentials in the folder ~/.cache/gargle
between R sessions?
1: Yes
2: No


Selection: 1


Please point your browser to the following url: 

https://accounts.google.com/o/oauth2/v2/auth?client_id=603366585132-frjlouoa3s2ono25d2l9ukvhlsrlnr7k.apps.googleusercontent.com&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive%20https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email&redirect_uri=https%3A%2F%2Fwww.tidyverse.org%2Fgoogle-callback%2F&response_type=code&state=548f88605ddaa21f7b16845523b931f3&access_type=offline&prompt=consent



Enter authorization code: eyJjb2RlIjoiNC8wQWRrVkxQellnNC1EdXJFYktmWm5vc2dDUjh6T1c2bmJhVDZuR1FycXpiZG41Rjh2VWQtVjMzajR0MVE2OGZmOUVvWERxUSIsInN0YXRlIjoiNTQ4Zjg4NjA1ZGRhYTIxZjdiMTY4NDU1MjNiOTMxZjMifQ==


In [2]:
install.packages("MASS")
install.packages("foreach")
install.packages("doParallel")
install.packages("quadprog")
install.packages("EnvStats")
install.packages("randcorr")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependency ‘iterators’


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependency ‘nortest’


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [3]:
rm(list = ls())
library(MASS) # for mvrnorm
library(quadprog) # for solve.QP (quadratic programming)
library(foreach) # for parallel loops
library(doParallel) # for parallel backend
library(EnvStats)
library(randcorr)

Loading required package: iterators

Loading required package: parallel


Attaching package: ‘EnvStats’


The following object is masked from ‘package:MASS’:

    boxcox


The following objects are masked from ‘package:stats’:

    predict, predict.lm


The following object is masked from ‘package:base’:

    print.default




In [4]:
vector_to_tensor <- function(vec, n_rows, n_cols) {
  if (length(vec) != n_rows * n_cols) {
    stop("Vector length must equal n_rows * n_cols")
  }
  matrix(vec, nrow = n_rows, ncol = n_cols, byrow = TRUE)
}

tensor_kernel <- function(X, Y, sigma = 1.0) {
  X_sq <- rowSums(X * X)
  Y_sq <- rowSums(Y * Y)
  D_sq <- outer(X_sq, Y_sq, "+") - 2 * (X %*% t(Y))
  K <- exp(-D_sq / (sigma^2))
  return(K)
}

kernel_uTu <- function(X, Y, u, sigma = 1.0) {
  W <- outer(u, u)
  X_sq <- rowSums(X * X)
  Y_sq <- rowSums(Y * Y)
  D_sq <- outer(X_sq, Y_sq, "+") - 2 * (X %*% t(Y))
  sum(exp(-D_sq / (sigma^2)) * W)
}

SVTDD_train <- function(tensors, sigma = 1.3, C = 1.5, n_cores = NULL) {
  N <- length(tensors)
  n <- nrow(tensors[[1]])
  p <- ncol(tensors[[1]])

  u <- rep(1, n)

  K_rows <- lapply(1:N, function(i) {
    sapply(1:N, function(j) kernel_uTu(tensors[[i]], tensors[[j]], u, sigma))
  })

  Kmat <- do.call(rbind, K_rows)

  Dmat <- 2 * Kmat + 1e-8 * diag(N)
  dvec <- diag(Kmat)

  Amat <- cbind(1, diag(N), -diag(N))
  bvec <- c(1, rep(0, N), rep(-C, N))

  qp_solution <- solve.QP(
    Dmat = Dmat,
    dvec = dvec,
    Amat = Amat,
    bvec = bvec,
    meq = 1
  )

  alpha <- qp_solution$solution

  alphaL <- qp_solution$Lagrangian

  sv_threshold <- 1e-5
  sv_indices <- which(alpha > sv_threshold)
  boundary_sv <- which(alpha > sv_threshold & alpha < (C - sv_threshold))

  alpha_K_alpha <- as.numeric(t(alpha) %*% Kmat %*% alpha)

  if (length(boundary_sv) > 0) {
    s <- boundary_sv[1]
    k_ss <- dvec[s]
    v <- Kmat[s, ]
    R_squared <- as.numeric(k_ss - 2 * sum(alpha * v) + alpha_K_alpha)
  } else {
    warning("No boundary support tensors found. Using alpha^T K alpha as radius.")
    R_squared <- alpha_K_alpha
  }

  model <- list(
    alpha = alpha,
    tensors = tensors,
    u = u,
    sigma = sigma,
    C = C,
    R_squared = R_squared,
    sv_indices = sv_indices,
    boundary_sv = boundary_sv,
    Kmat = Kmat,
    dvec = dvec,
    alpha_K_alpha = alpha_K_alpha
  )

  class(model) <- "SVTDD"
  return(model)
}

SVTDD_predict <- function(model, Z) {
  alpha <- model$alpha
  tensors <- model$tensors
  u <- model$u
  sigma <- model$sigma
  N <- length(tensors)

  k_zz <- kernel_uTu(Z, Z, u, sigma)

  k_zx <- vapply(1:N, function(j) {
    kernel_uTu(Z, tensors[[j]], u, sigma)
  }, numeric(1))

  d <- k_zz - 2 * sum(alpha * k_zx) + model$alpha_K_alpha

  return(d)
}

SignalProbability <- function(values, ucl) {
  sum(values > ucl) / length(values)
}


compute_loo_accuracy <- function(tensors, C_val, gamma_val) {
  N <- length(tensors)

  # Parallelized LOO loop using foreach
  results <- foreach(
    j = 1:N,
    .combine = "c",
    .packages = c("MASS", "quadprog"),
    .export = c(
      "SVTDD_train", "SVTDD_predict", "kernel_uTu",
      "tensor_kernel", "vector_to_tensor"
    ),
    .errorhandling = "pass"
  ) %dopar% {
    tryCatch(
      {
        X_train_loo <- tensors[-j]
        model_loo <- SVTDD_train(X_train_loo, sigma = gamma_val, C = C_val)

        TotalSVTDD1 <- vapply(seq_along(X_train_loo), function(i) {
          SVTDD_predict(model_loo, X_train_loo[[i]])
        }, numeric(1))

        B <- 5000
        h_values <- replicate(B, {
          resample_d <- sample(TotalSVTDD1, replace = TRUE)
          quantile(resample_d, 1 - 1 / 20)
        })

        UCL <- mean(h_values)
        # Predecir observacion dejada fuera
        tensor_test <- tensors[j]
        d_i <- SVTDD_predict(model_loo, tensor_test[[1]])

        if (d_i <= UCL * 1.08) { # 8% tolerance margin
          1L
        } else {
          0L
        }
      },
      error = function(e) 0L
    )
  }

  correct <- sum(unlist(results))
  accuracy <- correct / N
  return(accuracy)
}

select_optimal_parameters <- function(tensors,
                                      C_grid = c(0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1),
                                      gamma_grid = c(1.0, 1.1, 1.2, 1.3, 1.4, 1.5),
                                      n_cores = NULL) {

  # Configurar cluster paralelo
  if (is.null(n_cores)) {
    n_cores <- max(1, detectCores() - 1)
  }

  cl <- makeCluster(n_cores)
  registerDoParallel(cl)

  # Crear todas las combinaciones de parametros
  param_grid <- expand.grid(C = C_grid, gamma = gamma_grid)

  # Grid search paralelo sobre combinaciones de parametros
  results <- foreach(
    i = 1:nrow(param_grid),
    .combine = "rbind",
    .packages = c("MASS", "quadprog", "foreach", "doParallel"),
    .export = c(
      "compute_loo_accuracy", "SVTDD_train", "SVTDD_predict",
      "kernel_uTu", "tensor_kernel", "vector_to_tensor"
    )
  ) %dopar% {
    c_val <- param_grid$C[i]
    g <- param_grid$gamma[i]
    acc <- compute_loo_accuracy(tensors, C_val = c_val, gamma_val = g)

    data.frame(
      C = c_val,
      gamma = g,
      LOO_accuracy = acc
    )
  }

  # Detener cluster
  stopCluster(cl)

  # Encontrar mejores parametros
  results <- results[results$LOO_accuracy == max(results$LOO_accuracy),]
  best_idx <- which.max(results$gamma)
  best_accuracy <- results$LOO_accuracy[best_idx]
  best_C <- results$C[best_idx]
  best_gamma <- results$gamma[best_idx]

  return(list(
    optimal_C = best_C,
    optimal_gamma = best_gamma,
    optimal_accuracy = best_accuracy,
    results = results
  ))
}

In [5]:

for (Observation in c(100,150)){
  set.seed(123)
  # Parameters
  NumberVariable <- 200
  TensorRows <- 20
  TensorColumns <- 10
  mu <- rep(0, NumberVariable)

  fun <- function(i, j) (0.5)^(abs(i - j))
  SigmaCorr <- outer(1:NumberVariable, 1:NumberVariable, FUN = fun)

  X_train <- mvrnorm(Observation, mu, SigmaCorr)

  tensors_train <- lapply(1:nrow(X_train), function(i) {
    vector_to_tensor(X_train[i, ], TensorRows, TensorColumns)
  })

  USE_LOO <- TRUE

  if (USE_LOO) {
    optimal_params <- select_optimal_parameters(
      tensors_train
    )
    OPTIMAL_C <- optimal_params$optimal_C
    OPTIMAL_GAMMA <- optimal_params$optimal_gamma

  } else {
    OPTIMAL_C <- 0.5
    OPTIMAL_GAMMA <- 1
  }
  svtdd_model <- SVTDD_train(tensors_train, sigma = OPTIMAL_GAMMA, C=OPTIMAL_C)

  # Configurar cluster paralel
  n_cores <- detectCores() - 1
  cl <- makeCluster(n_cores)
  registerDoParallel(cl)

  TotalSVTDD1 <- foreach(
    iter = 1:10000,
    .combine = c,
    .packages = c("MASS"),
    .export = c(
      "Observation", "mu", "SigmaCorr", "TensorRows", "TensorColumns",
      "vector_to_tensor", "SVTDD_predict", "svtdd_model", "kernel_uTu"
    )
  ) %dopar% {
    Data <- mvrnorm(Observation, mu, SigmaCorr)

    tensors_test_ooc <- lapply(1:nrow(Data), function(i) {
      vector_to_tensor(Data[i, ], TensorRows, TensorColumns)
    })

    results <- sapply(1:Observation, function(i) {
      SVTDD_predict(svtdd_model, tensors_test_ooc[[i]])
    })
    results
  }

  UCL <- qemp(p = 1 - 1 / 20, obs = TotalSVTDD1)
  stopCluster(cl)

  # Configurar cluster paralelo
  n_cores <- detectCores() - 1
  cl <- makeCluster(n_cores)
  registerDoParallel(cl)

  ArrayRhoShift <- list(
    mu,
    rep(10 / 100, NumberVariable),
    rep(5 / 100, NumberVariable),
    rep(10 / 100, NumberVariable),
    rep(15 / 100, NumberVariable),
    rep(25 / 100, NumberVariable),
    rep(50 / 100, NumberVariable),
    rep(60 / 100, NumberVariable),
    rep(75 / 100, NumberVariable),
    rep(90 / 100, NumberVariable),
    rep(1, NumberVariable)
  )


  Inverse <- solve(SigmaCorr)
  for (outlierValues in c(0.05,0.1,0.2)){
    MatrixDelta <- matrix(, ncol = 2)
    percentoutliers <- outlierValues
    ih <- 1
    for (meanShift in ArrayRhoShift)
    {
      NumOutliers <- floor(percentoutliers * Observation)
      is_baseline <- identical(mu, meanShift)

      TotalSVTDD <- foreach(
        iter = 1:10000,
        .combine = c,
        .packages = c("MASS"),
        .export = c(
          "Observation", "mu", "SigmaCorr", "TensorRows", "TensorColumns",
          "vector_to_tensor", "SVTDD_predict", "svtdd_model", "kernel_uTu",
          "NumOutliers", "meanShift", "is_baseline"
        )
      ) %dopar% {
        if (is_baseline) {
          Data <- mvrnorm(Observation, mu = mu, Sigma = SigmaCorr)
          tensors_test_ooc <- lapply(1:nrow(Data), function(i) {
            vector_to_tensor(Data[i, ], TensorRows, TensorColumns)
          })
          results <- sapply(1:Observation, function(i) {
            SVTDD_predict(svtdd_model, tensors_test_ooc[[i]])
          })
        } else {
          Data <- mvrnorm(Observation - NumOutliers, mu = mu, Sigma = SigmaCorr)
          if (NumOutliers >= 1) {
            DataOutlier <- mvrnorm(NumOutliers, mu = meanShift, Sigma = SigmaCorr)
            Data <- rbind(Data, DataOutlier)
          }
          tensors_test_ooc <- lapply(1:nrow(DataOutlier), function(i) {
            vector_to_tensor(DataOutlier[i, ], TensorRows, TensorColumns)
          })
          results <- sapply(1:NumOutliers, function(i) {
            SVTDD_predict(svtdd_model, tensors_test_ooc[[i]])
          })
        }
        results
      }

      SignalProbabilityT2OutliersMRCD <- SignalProbability(TotalSVTDD, UCL)
      MatrixDelta <- rbind(MatrixDelta, c(ih, SignalProbabilityT2OutliersMRCD))
      ih <- ih + 1
    }
    url = paste("/content/SinalprobabilitySVTDDNormalMC",Observation,"x",NumberVariable,"x",outlierValues,".RData", sep = "")
    save.image(url)
    drive_upload(url, path = "Colab10000SVTDDNormal/")
  }

  stopCluster(cl)
}

Warning message in e$fun(obj, substitute(ex), parent.frame(), e$data):
“already exporting variable(s): Observation, mu, SigmaCorr, TensorRows, TensorColumns, vector_to_tensor, SVTDD_predict, svtdd_model, kernel_uTu”
Warning message in e$fun(obj, substitute(ex), parent.frame(), e$data):
“already exporting variable(s): Observation, mu, SigmaCorr, TensorRows, TensorColumns, vector_to_tensor, SVTDD_predict, svtdd_model, kernel_uTu, NumOutliers, meanShift, is_baseline”
Warning message in e$fun(obj, substitute(ex), parent.frame(), e$data):
“already exporting variable(s): Observation, mu, SigmaCorr, TensorRows, TensorColumns, vector_to_tensor, SVTDD_predict, svtdd_model, kernel_uTu, NumOutliers, meanShift, is_baseline”
Warning message in e$fun(obj, substitute(ex), parent.frame(), e$data):
“already exporting variable(s): Observation, mu, SigmaCorr, TensorRows, TensorColumns, vector_to_tensor, SVTDD_predict, svtdd_model, kernel_uTu, NumOutliers, meanShift, is_baseline”
Warning message in e$fun(